In [36]:
import numpy as np
import re
import json
from pathlib import Path

In [38]:
class PID_group(object):
    
    def __init__(self, pid: str):
        self.pid = pid
        self.parameters = []
        self.init = []

    def add_parameter(self, name: str, init: str, unit: str, expr: str):
        self.parameters.append({})
        self.parameters[-1]["name"] = name
        if init not in self.init:
            self.init.append(init)
        self.parameters[-1]["unit"] = unit
        self.parameters[-1]["expr"] = process_expression(expr,self.pid)

    def format_init(self):
        unique_init = np.unique(self.init)
        formatted_inits = []
        for init in unique_init:
            formatted_inits.append('ATSH'+init)
        self.formatted_init_string = ';'.join(formatted_inits)

    def make_json_dict(self):
        self.format_init()
        self.json_dict = {"pid" : self.pid, 
                          "pid_init" : self.formatted_init_string,
                          "parameters" : self.parameters}
        return self.json_dict

In [50]:
def process_expression(expr: str, pid: str):
    if(len(pid) == 4 or len(pid) == 5):
        return translate(expr, 0)
    elif(len(pid) == 6 or len(pid) == 7):
        return translate(expr, 1)
    else:
        raise ValueError 

def translate(expr: str, offset: int):
    print(f"Init. expression: {expr}")
    # add brackets
    expr = re.sub(r"\b(?<!:)([A-Za-z]?[A-Za-z]):([A-Za-z]?[A-Za-z])",r"[\1:\2",expr)
    expr = re.sub(r"([A-Za-z]?[A-Za-z]):([A-Za-z]?[A-Za-z])(?!:)\b",r"\1:\2]",expr)
    expr = re.sub(r"\b([A-Za-z]?[A-Za-z]):([A-Za-z]?[A-Za-z]:)+([A-Za-z]?[A-Za-z])\b",r"\1:\3",expr)
    torque_column = re.compile(r"(\b[A-Za-z]?[A-Za-z]\b)")
    cols_to_replace = torque_column.findall(expr)
    split_expression = torque_column.split(expr)
    new_expr = ''
    for part in split_expression:
        if part in cols_to_replace:
            if len(part) == 2:
                new_expr += replace_double_letters(part,part,offset)
            elif len(part) == 1:
                new_expr += replace_single_letters(part,part,offset)
        else:
            new_expr += part
    # handle casts
    new_expr = re.sub(r"SIGNED\(B([0-9]+)\)",r"(S\1)",new_expr,flags=re.IGNORECASE)
    new_expr = re.sub(r"SIGNED\(\[B([0-9]+):B([0-9]+)\]\)",r"([S\1:S\2])",new_expr,flags=re.IGNORECASE)
    new_expr = re.sub(r"INT\d+\(([][B0-9:]+)\)",r"(\1)",new_expr,flags=re.IGNORECASE)
    new_expr = re.sub(r"{(B[0-9]+:[0-7])}",r"\1",new_expr,flags=re.IGNORECASE)
    # translate bitwise operators
    new_expr = re.sub(r"<",r"<<",new_expr)
    new_expr = re.sub(r">",r">>",new_expr)
    print(f"Final expression: {new_expr}\n")
    return new_expr

def replace_single_letters(letter: str, expr: str, offset: int):
    val = ord(letter.lower())-ord('a') + offset + 4
    frame_type = (val-1) // 7 
    val += frame_type
    # re here is overkill and based on a previous attempt
    return re.sub(rf"\b{letter}\b",f"B{val}",expr)
    
def replace_double_letters(letters: str, expr: str, offset: int):
    val = ord(letters.lower()[1])-ord('a')+4+offset+26
    frame_type = (val-1) // 7 
    val += frame_type
    return re.sub(rf"\b{letters}\b",f"B{val}",expr)

In [55]:
fname = '/home/trh/Packages/Hyundai-Ioniq-5-Torque-Pro-PIDs/TorqueIONIQ5AWD74kWh.csv'
pids = np.loadtxt(fname,delimiter=',',dtype='str',comments='~',skiprows=1)

/tmp/ipykernel_125584/482868756.py:2: UserWarning: Input line 2 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  pids = np.loadtxt(fname,delimiter=',',dtype='str',comments='~',skiprows=1)


In [56]:
pid_groups = []
for line in pids:
    if("val" in line[3] or len(line[2]) == 0):
        continue
    pid = line[2].lstrip('0x') 
    pid_in_list = [group.pid == pid for group in pid_groups]
    if(any(pid_in_list)):
        group = np.array(pid_groups)[pid_in_list][0]
    else:
        group = PID_group(pid)
        pid_groups.append(group)
    group.add_parameter(line[0], line[7], line[6], line[3])
# print([group.pid for group in pid_groups])
# print([group.parameters for group in pid_groups])

Init. expression: e/50
Final expression: B10/50

Init. expression: f/50
Final expression: B11/50

Init. expression: g/50
Final expression: B12/50

Init. expression: h/50
Final expression: B13/50

Init. expression: i/50
Final expression: B14/50

Init. expression: j/50
Final expression: B15/50

Init. expression: k/50
Final expression: B17/50

Init. expression: l/50
Final expression: B18/50

Init. expression: m/50
Final expression: B19/50

Init. expression: n/50
Final expression: B20/50

Init. expression: o/50
Final expression: B21/50

Init. expression: p/50
Final expression: B22/50

Init. expression: q/50
Final expression: B23/50

Init. expression: r/50
Final expression: B25/50

Init. expression: s/50
Final expression: B26/50

Init. expression: t/50
Final expression: B27/50

Init. expression: u/50
Final expression: B28/50

Init. expression: v/50
Final expression: B29/50

Init. expression: w/50
Final expression: B30/50

Init. expression: x/50
Final expression: B31/50

Init. expression: y/

In [57]:
json_dict = {"pids" : []}
for group in pid_groups:
    json_dict["pids"].append(group.make_json_dict())

In [58]:
new_fname = Path(fname).with_suffix('.json')
with open(new_fname,'w') as f:
    json.dump(json_dict, f, indent=2)